# BioTek Synergy HT

```{device-card} biotek-synergy-ht
```

| Property | Supported behavior |
| --- | --- |
| Interface | USB FTDI serial adapter; 9600 baud, 8 data bits, no parity, 2 stop bits |
| Assay | Single-wavelength endpoint absorbance |
| Plates | Reader's built-in 24- and 96-well presets |
| Results | `AbsorbanceResult` with OD data, wavelength, temperature, and timestamp |
| Additional operations | Tray opening/closing, temperature measurement, incubation, shaking before reads |

**Untested on hardware.** Offline coverage uses Synergy HT basecode `7090202`, firmware `2.24`.
Please report successful hardware verification on the PyLabRobot forum before removing this warning.
Other basecodes are rejected; other firmware versions must report the required capabilities and
remain unverified. This driver does not support Synergy H1, HTX, or Epoch.

Fluorescence, luminescence, standalone shaking, kinetics, 384-well plates, and custom geometry are not implemented.


## Connection and physical setup

Install `pylabrobot[ftdi]` and the platform's libftdi runtime as described in the
[installation guide](../../getting-started/installation.md). Connect the reader's serial interface
through its USB FTDI adapter and power on the reader. Wait for normal boot to finish. Close Gen5
and other applications using the same adapter.

Use an unlidded, optically suitable plate matching the reader's built-in preset. A matching well
count alone does not establish matching geometry. This driver does not restore erased reader
configuration or recover a reader stuck during boot.

Import the driver. All following cells communicate with the instrument when you execute them.


In [ ]:
from pylabrobot.agilent.biotek import SynergyHT

## Connect

Create the connection. If several FTDI adapters are attached, set `device_id` to the adapter's USB
serial number, which can differ from the reader's serial number. The measurement timeout is in seconds.


In [ ]:
reader = SynergyHT(device_id=None, read_timeout=180)

Initialize the connection and check the model and required protocol capabilities.

In [ ]:
await reader.setup()

Inspect the reported identity and allowed incubation setpoints.

In [ ]:
(reader.serial_number, reader.version, reader.temperature_range)

## Load a plate

Open the tray. Ensure its travel path is clear.


In [ ]:
await reader.open()

Place a compatible, unlidded 96-well plate on the tray, with A1 aligned to the tray marking. Then close the tray.

In [ ]:
await reader.close()

## Read absorbance

Read the full plate at 600 nm. Pass the plate type as a string: `"96-well"` or `"24-well"`,
matching the built-in preset for the loaded plate. The instrument validates its wavelength range.

Like the other BioTek readers, this returns a list of `AbsorbanceResult` objects. An HT endpoint
read produces one result. Its `.data` matrix is indexed `[row][column]`: A1 is `[0][0]`, B1 is `[1][0]`.
Values are raw OD, without blank subtraction or pathlength correction. Unselected wells are `None`,
and instrument overflow is `NaN`. Include suitable blanks and standards for your assay.

In [ ]:
results = await reader.read_absorbance(wavelength=600, plate_type="96-well")

Inspect the first row of the endpoint result.

In [ ]:
results[0].data[0]

Inspect the measurement metadata. The wavelength is in nm, and the reported incubator temperature
is in °C (`None` if unavailable). The timestamp is the host's local time when the read completed.

In [ ]:
(results[0].wavelength, results[0].temperature, results[0].timestamp)

## Shake before reading

Shake the plate for 10 seconds before measuring absorbance. `shake_duration` is a whole number of
seconds from 0 to 65535; zero (the default) disables shaking. `shake_intensity` selects one of the
instrument's four levels, `0` through `3`. These levels are not frequencies in Hz or RPM.

The tray must be closed. The call waits for both shaking and the measurement to finish, allowing
`shake_duration` extra seconds beyond `read_timeout`. Shaking settings apply only to this call;
the next read uses its own settings. Standalone shaking is not supported by the legacy Gen5 driver.

In [ ]:
shaken = await reader.read_absorbance(
  wavelength=600,
  plate_type="96-well",
  shake_duration=10,
  shake_intensity=2,
)

## Select wells

For a partial read, provide well names. The instrument measures their bounding rectangle;
`selected[0].data` contains the full-plate matrix, with `None` for all wells you did not request.

In [ ]:
selected = await reader.read_absorbance(600, "96-well", wells=["A1", "B2"])

## Optional incubation

Enable incubation at 37 °C. This returns when the setpoint is accepted, before the plate has
necessarily reached that temperature. Check `reader.temperature_range` for this instrument's limits.


In [ ]:
await reader.set_temperature(37)

Read the current incubator temperature in °C. This is not a measurement of the liquid temperature.

In [ ]:
temperature = await reader.get_temperature()

Disable incubation when it is no longer needed.

In [ ]:
await reader.stop_heating()

## Unload and disconnect

Open the tray and remove the plate.


In [ ]:
await reader.open()

After removing the plate, close the empty tray.

In [ ]:
await reader.close()

Disconnect. `stop()` does not disable incubation or abort a measurement. A failed or cancelled
exchange closes the connection. Ensure the reader is idle before calling `setup()` to reconnect.

In [ ]:
await reader.stop()

## Offline validation

The tests replay complete 24- and 96-well command sequences, parse 15 measurement frames, and compare
six setup packets with isolated execution of Gen5 3.16.10's legacy serializer. All captured absorbance
values are zero. Signed nonzero values, overflow, and the milli-OD conversion are covered separately;
the 0.001 conversion factor is present in Gen5's result parser.

The capture named `startSoftwareMakeNewExperiment384Well.pcap` selects the 96-well preset and returns
96 measurements, so it does not validate 384-well support. The captures named for partial well
selections also read full plates. Selection bounds are implemented from Gen5's command construction
and checked with synthetic responses.

Capture filenames, SHA-256 hashes, packet numbers, extracted UART data, and serializer vectors are
kept in the driver's `test_data/synergy_ht*.json` fixtures. The driver needs neither Gen5 nor its DLLs
at runtime. Actual optical performance and mechanical compatibility still need hardware testing.


Shaking coverage compares all 12 setup packets from `multiShakeRead.pcap`, including 8- and
10-second durations and all four intensity levels, and replays a complete shake/read sequence.
Additional setup vectors come from isolated execution of Gen5's serializer and cover two-byte
durations and the maximum encoded duration. These checks do not verify physical shaking performance.
